# Open-Source Models, Embeddings & Vector Search

**Day 2** | Topics from the course outline:
- The Hugging Face open-source ecosystem
- Working with HF models: summarization, sentiment analysis, question answering
- Embeddings and semantic similarity (cosine distance)
- Vector databases (FAISS) and retrieval pipelines
- **Lab:** Build an embedding-based semantic document search system

**What you will build:**
A semantic search system that encodes support tickets as vectors, stores them in FAISS, and retrieves the most relevant tickets for a natural-language query — without keyword matching.

**Prerequisites:** Day 1 complete. Python 3.11, `.venv-student`. Additional packages: see Setup below.


## 1. Where open-source models fit — from APIs to local inference

```
  DAY 1 (API models)                      DAY 2 (open-source / local)
  ─────────────────                       ──────────────────────────
  ┌──────────┐   HTTP    ┌──────────┐    ┌──────────────┐   Python   ┌──────────────┐
  │ Your     │──────────▶│ OpenAI   │    │ Your         │──────────▶│ HuggingFace  │
  │ Python   │  REST     │ API      │    │ Python       │  direct   │ model        │
  │ code     │◀──────────│ (remote) │    │ code         │◀──────────│ (local)      │
  └──────────┘   JSON    └──────────┘    └──────────────┘  tensors  └──────────────┘
                                               │
                                               ▼
                                         ┌──────────────┐
                                         │ FAISS        │
                                         │ vector index │
                                         │ (local)      │
                                         └──────────────┘
```

**Day 1:** You sent requests to a remote API and parsed JSON responses.
**Day 2:** You load models locally with `transformers` and `sentence-transformers`. No API key needed for HF models — they run on your machine.

| | API models (Day 1) | Open-source models (Day 2) |
|---|---|---|
| **Where it runs** | Remote server (OpenAI, Azure) | Your machine or server |
| **Cost** | Per-token pricing | Free (compute cost only) |
| **Setup** | API key + `requests` | `pip install transformers` + model download |
| **Latency** | Network round-trip | Local inference (no network) |
| **Customization** | Prompt engineering | Fine-tuning, model selection |
| **Privacy** | Data sent to third party | Data stays local |


## 2. Setup

Install the additional packages for Day 2 (run once). If you already have them, this cell is fast.

```bash
pip install transformers sentence-transformers faiss-cpu torch
```

> **Download note:** The first time you use a model, `transformers` downloads it (~80–800 MB depending on model). This happens once per model; subsequent runs use the cached version.


In [2]:
import warnings
warnings.filterwarnings("ignore")

deps_ok = True
for pkg, imp in [("transformers", "transformers"), ("sentence-transformers", "sentence_transformers"),
                 ("faiss-cpu", "faiss"), ("torch", "torch"), ("numpy", "numpy")]:
    try:
        __import__(imp)
    except ImportError:
        print(f"Missing: {pkg}  →  pip install {pkg}")
        deps_ok = False

if deps_ok:
    import numpy as np
    import faiss
    import torch
    from transformers import pipeline as hf_pipeline
    from sentence_transformers import SentenceTransformer
    print("All Day 2 dependencies loaded.")
    print(f"  torch: {torch.__version__}, faiss: {faiss.__version__ if hasattr(faiss, '__version__') else 'OK'}")
else:
    print("\nInstall missing packages, restart kernel, then re-run this cell.")


All Day 2 dependencies loaded.
  torch: 2.10.0+cpu, faiss: 1.13.2


## 3. The Hugging Face ecosystem

**Hugging Face Hub** hosts 500k+ models, 100k+ datasets, and community tools — all open-source.

Key concepts:
- **Pipeline:** One-line interface to load a pre-trained model and run inference. Handles tokenization, model loading, and output parsing.
- **Model:** A pre-trained neural network (e.g. `distilbart-cnn` for summarization, `distilbert` for sentiment).
- **Task:** What the model does — `summarization`, `sentiment-analysis`, `question-answering`, `feature-extraction`, etc.

```python
from transformers import pipeline
summarizer = pipeline("summarization", model="Falconsai/text_summarization")
result = summarizer("Your long text here...", max_length=80)
```

The `pipeline()` function is the main entry point. It:
1. Downloads the model (first run only).
2. Tokenizes your input.
3. Runs inference.
4. Returns structured output.


### Task 1 — Text summarization with Hugging Face

**Use case:** Same as Day 1 (executive summaries, report compression) — but running **locally** instead of calling an API.

**Model:** `Falconsai/text_summarization` — a T5-small model fine-tuned for summarization. Small (~240 MB) and fast on CPU.


In [3]:
summarizer = hf_pipeline("summarization", model="Falconsai/text_summarization")

report = (
    "The company reported strong Q3 results with revenue increasing 12% year-over-year, "
    "primarily driven by enterprise contracts in the Asia-Pacific region. Customer satisfaction "
    "scores improved from 4.1 to 4.5 out of 5. Operations expanded to Latin America and the "
    "Middle East with new local offices. However, two product launches were delayed to Q4 due "
    "to dependencies on the new authentication platform. Engineering headcount grew by 15%, "
    "while cloud infrastructure costs increased 8% but unit cost per customer dropped 3%. "
    "The churn rate decreased from 5.2% to 4.1%, the lowest in company history."
)

result = summarizer(report, max_length=80, min_length=20, do_sample=False)
print("Original length:", len(report.split()), "words")
print("Summary:", result[0]["summary_text"])


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original length: 91 words
Summary: Customer satisfaction scores improved from 4.1 to 4.5 out of 5 . Operations expanded to Latin America and the Middle East with new local offices .


### Task 2 — Sentiment analysis

**Use case:** Classify customer feedback, support tickets, or reviews as positive/negative. Useful for routing (negative → priority queue) or dashboards.

**Model:** Default `distilbert-base-uncased-finetuned-sst-2-english` (~260 MB).


In [4]:
classifier = hf_pipeline("sentiment-analysis")

texts = [
    "Great product! Shipping was fast and quality exceeded expectations.",
    "Terrible experience. The app crashes every time I open it.",
    "It works fine but nothing special. Average product overall.",
    "I love the new dashboard feature — makes reporting so much easier!",
    "I want my money back. This is completely broken.",
]

print("Sentiment analysis:")
for text in texts:
    result = classifier(text)[0]
    print(f"  {result['label']:8s} ({result['score']:.3f})  {text[:60]}")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Sentiment analysis:
  POSITIVE (1.000)  Great product! Shipping was fast and quality exceeded expect
  NEGATIVE (1.000)  Terrible experience. The app crashes every time I open it.
  NEGATIVE (0.539)  It works fine but nothing special. Average product overall.
  POSITIVE (0.999)  I love the new dashboard feature — makes reporting so much e
  NEGATIVE (1.000)  I want my money back. This is completely broken.


### Task 3 — Question answering (extractive)

**Use case:** Given a context paragraph, extract the answer to a specific question. This is the foundation for retrieval-augmented systems (Day 3).

**How it differs from chat:** Extractive QA **selects a span** from the context — it does not generate new text.


In [5]:
qa_pipeline = hf_pipeline("question-answering")

context = (
    "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search "
    "and clustering of dense vectors. It contains algorithms that search in sets of vectors "
    "of any size, up to ones that do not fit in RAM. FAISS is written in C++ with complete "
    "wrappers for Python. It was developed by Facebook AI Research."
)

questions = [
    "What is FAISS?",
    "Who developed FAISS?",
    "What programming language is FAISS written in?",
]

for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"  Q: {q}")
    print(f"  A: {result['answer']}  (score: {result['score']:.3f})")
    print()


No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


  Q: What is FAISS?
  A: Facebook AI Similarity Search  (score: 0.367)

  Q: Who developed FAISS?
  A: Facebook AI Research  (score: 0.873)

  Q: What programming language is FAISS written in?
  A: C++  (score: 0.971)



### When to use API models vs local models

| Criterion | API (Day 1) | Local / open-source (Day 2) |
|-----------|-------------|---------------------------|
| **Best for** | Complex generation, multi-turn chat, vision | Classification, extraction, embeddings, search |
| **Speed** | Network-bound (0.5–5s) | CPU/GPU-bound (0.01–2s) |
| **Cost at scale** | Per-token (can be expensive) | Compute only (predictable) |
| **Data privacy** | Data leaves your network | Data stays local |
| **Customization** | Prompt engineering | Fine-tuning, model swapping |
| **Maintenance** | Provider handles updates | You manage model versions |

**In practice:** Many production systems use **both** — local models for high-volume tasks (classification, embeddings) and API models for complex generation.


## 4. Embeddings and semantic similarity

**What is an embedding?**
An embedding is a dense vector (list of numbers) that represents the *meaning* of a piece of text. Texts with similar meaning have vectors that are close together in vector space.

```
  "login problem"     →  [0.12, -0.45, 0.78, ..., 0.33]   ─┐
  "can't sign in"     →  [0.11, -0.43, 0.76, ..., 0.31]   ─┤ close together (similar meaning)
  "billing question"  →  [-0.67, 0.22, -0.15, ..., 0.88]   ─┘ far apart (different meaning)
```

**Why it matters:** Keyword search fails when the query uses different words than the document ("login problem" won't match "authentication error"). Embedding-based search finds matches by **meaning**, not keywords.

**Model:** `all-MiniLM-L6-v2` — a small (80 MB), fast sentence-transformer model that produces 384-dimensional embeddings.


In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The customer cannot log in to their account",
    "User authentication is failing with error 500",
    "Invoice shows a double charge for March",
    "The analytics dashboard is loading very slowly",
    "Application crashes on iOS 17 when opening notifications",
]

embeddings = model.encode(sentences)
print(f"Encoded {len(sentences)} sentences → shape: {embeddings.shape}")
print(f"Each sentence becomes a {embeddings.shape[1]}-dimensional vector")
print(f"\nFirst embedding (first 10 values): {embeddings[0][:10].round(3)}")


Encoded 5 sentences → shape: (5, 384)
Each sentence becomes a 384-dimensional vector

First embedding (first 10 values): [ 0.003  0.024 -0.045 -0.088 -0.075 -0.026  0.051 -0.051  0.034 -0.004]


### Cosine similarity — measuring semantic closeness

**Cosine similarity** measures the angle between two vectors. Range: -1 (opposite) to 1 (identical).

- **1.0** = identical meaning
- **0.7–0.9** = very similar
- **0.3–0.6** = somewhat related
- **< 0.3** = unrelated


In [8]:
from numpy.linalg import norm


def cosine_sim(a, b):
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (norm(a) * norm(b)))


print("Pairwise similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_sim(embeddings[i], embeddings[j])
        marker = " ← similar!" if sim > 0.5 else ""
        print(f"  {sim:.3f}  [{i}] vs [{j}]: "
              f"\"{sentences[i][:40]}\" vs \"{sentences[j][:40]}\"{marker}")


Pairwise similarities:
  0.497  [0] vs [1]: "The customer cannot log in to their acco" vs "User authentication is failing with erro"
  0.108  [0] vs [2]: "The customer cannot log in to their acco" vs "Invoice shows a double charge for March"
  0.124  [0] vs [3]: "The customer cannot log in to their acco" vs "The analytics dashboard is loading very "
  0.099  [0] vs [4]: "The customer cannot log in to their acco" vs "Application crashes on iOS 17 when openi"
  0.092  [1] vs [2]: "User authentication is failing with erro" vs "Invoice shows a double charge for March"
  0.084  [1] vs [3]: "User authentication is failing with erro" vs "The analytics dashboard is loading very "
  0.116  [1] vs [4]: "User authentication is failing with erro" vs "Application crashes on iOS 17 when openi"
  0.043  [2] vs [3]: "Invoice shows a double charge for March" vs "The analytics dashboard is loading very "
  0.069  [2] vs [4]: "Invoice shows a double charge for March" vs "Application crashes on iOS 17 whe

## 5. Vector databases — FAISS

**Problem:** With 10,000+ documents, computing cosine similarity against every document is slow. A vector database builds an **index** for fast approximate nearest-neighbor search.

**FAISS** (Facebook AI Similarity Search):
- Written in C++, Python bindings.
- Supports exact (`IndexFlatL2`) and approximate (`IndexIVFFlat`, `IndexHNSW`) search.
- Handles millions of vectors efficiently.

```
  Documents  →  Encode  →  Vectors  →  FAISS Index  →  Query  →  Top-K results
                (model)     (float32)    (build once)    (fast)
```


In [9]:
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)

embeddings_f32 = embeddings.astype("float32")
index.add(embeddings_f32)

print(f"FAISS index: {index.ntotal} vectors, dimension {dimension}")
print(f"Index type: IndexFlatL2 (exact L2 distance)")


FAISS index: 5 vectors, dimension 384
Index type: IndexFlatL2 (exact L2 distance)


In [10]:
def search(query_text, k=3):
    """Encode query, search FAISS index, return top-k results with distances."""
    q_emb = model.encode([query_text]).astype("float32")
    distances, indices = index.search(q_emb, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({"rank": len(results) + 1, "index": int(idx),
                         "distance": float(dist), "text": sentences[idx]})
    return results


queries = ["password reset issue", "slow application performance", "payment problem"]
for q in queries:
    print(f"\nQuery: \"{q}\"")
    for r in search(q):
        print(f"  {r['rank']}. [{r['distance']:.3f}] {r['text']}")



Query: "password reset issue"
  1. [0.992] The customer cannot log in to their account
  2. [1.107] User authentication is failing with error 500
  3. [1.793] Invoice shows a double charge for March

Query: "slow application performance"
  1. [1.001] The analytics dashboard is loading very slowly
  2. [1.526] Application crashes on iOS 17 when opening notifications
  3. [1.883] User authentication is failing with error 500

Query: "payment problem"
  1. [1.301] The customer cannot log in to their account
  2. [1.390] Invoice shows a double charge for March
  3. [1.556] User authentication is failing with error 500


## 6. Lab — Semantic document search over support tickets

**Goal:** Build a search system that finds the most relevant support tickets for a natural-language query, using the `tickets.csv` file from Day 1.

**Pipeline:**
```
  tickets.csv  →  Encode (sentence-transformers)  →  FAISS index  →  Query  →  Top-K tickets
```

**Steps:**
1. Load tickets from CSV.
2. Encode each ticket (subject + body) as a vector.
3. Build a FAISS index.
4. Write a search function.
5. Test with several queries.


In [11]:
import csv

documents = []
with open("tickets.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        text = f"{row['subject']}: {row['body']}"
        documents.append({"id": row["ticket_id"], "text": text,
                          "subject": row["subject"], "priority": row["priority"]})

print(f"Loaded {len(documents)} tickets from tickets.csv")
for d in documents[:3]:
    print(f"  #{d['id']} [{d['priority']}] {d['text'][:70]}...")


Loaded 18 tickets from tickets.csv
  #1 [high] Login failed: Getting error 500 when I click Forgot password. Tried fr...
  #2 [high] Invoice discrepancy: My last invoice shows double charge for the premi...
  #3 [medium] Export to PDF: We need an option to export reports to PDF for our mont...


In [12]:
doc_texts = [d["text"] for d in documents]
doc_embeddings = model.encode(doc_texts).astype("float32")

doc_index = faiss.IndexFlatL2(doc_embeddings.shape[1])
doc_index.add(doc_embeddings)

print(f"Indexed {doc_index.ntotal} tickets ({doc_embeddings.shape[1]}-dim vectors)")


Indexed 18 tickets (384-dim vectors)


In [13]:
def search_tickets(query, k=3):
    """Search support tickets by meaning. Returns top-k matches."""
    q_emb = model.encode([query]).astype("float32")
    distances, indices = doc_index.search(q_emb, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        doc = documents[idx]
        results.append({**doc, "distance": round(float(dist), 3)})
    return results


test_queries = [
    "user cannot log in",
    "billing or payment issue",
    "application is slow",
    "security certificate problem",
    "need help getting started",
    "API integration errors",
]

for q in test_queries:
    print(f"\nQuery: \"{q}\"")
    for r in search_tickets(q, k=3):
        print(f"  [{r['distance']:.3f}] #{r['id']} [{r['priority']}] {r['subject']}")



Query: "user cannot log in"
  [1.079] #1 [high] Login failed
  [1.388] #9 [high] Permission denied
  [1.505] #6 [high] SSO not working

Query: "billing or payment issue"
  [1.086] #2 [high] Invoice discrepancy
  [1.232] #10 [low] Billing address update
  [1.481] #5 [low] Renewal date

Query: "application is slow"
  [1.240] #7 [medium] Dashboard slow
  [1.557] #4 [high] API rate limit
  [1.640] #12 [high] Mobile app crash

Query: "security certificate problem"
  [0.852] #15 [high] Custom domain SSL
  [1.370] #6 [high] SSO not working
  [1.498] #1 [high] Login failed

Query: "need help getting started"
  [1.445] #17 [low] Onboarding help
  [1.808] #5 [low] Renewal date
  [1.825] #9 [high] Permission denied

Query: "API integration errors"
  [1.014] #4 [high] API rate limit
  [1.349] #11 [medium] Webhook failures
  [1.508] #1 [high] Login failed
